# Lesson 000.09 -- Reading with One Eye Open: Recursive-Descent Parsing

AuthorityEffect: None.

> **Claim-typing legend**
> - **[Pinned]**: reproduced from verified checked-in Grammar 000 specifications (Parsing Laws 5 and 6).
> - **[Executed]**: verified by deterministic Python execution in this notebook.
> - **[Declared]**: pedagogical convention or beginner framing.
> - **[Open]**: questions left for future exploration.

## 1. The Big Idea: One-Character Lookahead and Reserved Prefixes

[Pinned] Why is Grammar 000 called "reading with one eye open"?
Because a recursive-descent parser requires **at most one choice inspection** at every position in the wire stream:
1. **Application Check:** Look at the next 3 characters. If they match the literal reserved prefix `op<`, consume `op<` and parse children until `>`.
2. **Atom / Diff Check:** If the next character is `<`, consume `<` and read an alphanumeric word run until `>`. If the word is empty, produce `Diff()`; otherwise produce `Atom(word)`.
3. **Syntax Error:** If the next character is neither `op<` nor `<`, the parser halts immediately with a `GrammarError` flagging bare, unwrapped content.

Because every child is self-delimiting, the parser needs zero global bracket counting or complex backtracking.

## 2. Maximal Recognition: `<op>` vs. `op<...>`

[Pinned] Parsing Law 6 requires maximal recognition of the reserved prefix:
- `op<>` begins with `op<`, so it is parsed as an empty application `App(())`.
- `<op>` begins with `<`, so it is parsed as an `Atom(word="op")`.
- `op<<0>>` is an application holding child `<0>`.

### Error Localization with `GrammarError`
When an input violates grammar rules, `GrammarError` records:
- The exact zero-indexed character position where failure was detected (`e.position`).
- An honest mechanical diagnostic reason (`e.reason`).

In [ ]:
# [Pinned] Environment Preflight & Project Root Resolution
# Pinned revision: 3f3399766108adc4c4e7615dce2573a57be57dd6
# Supported environment: Python 3.12+ (e.g. py -3.12 or pixi run)
# Project install: pip install -e .
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().resolve()
found = False
while ROOT_DIR.parent != ROOT_DIR:
    if (
        (ROOT_DIR / "omega_engine_pfp").exists()
        or (ROOT_DIR / ".git").exists()
        or (ROOT_DIR / "pixi.toml").exists()
    ):
        if (ROOT_DIR / "omega_engine_pfp").exists():
            found = True
            break
    ROOT_DIR = ROOT_DIR.parent

if found and str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
elif not found:
    raise RuntimeError(
        "Could not locate 'omega_engine_pfp' root. "
        "Ensure you execute from a repository checkout or install via: pip install -e ."
    )

from omega_engine_pfp.domain.tower import grammar

# 1. Distinguishing <op> from op<>
atom_op = grammar.parse("<op>")
app_op = grammar.parse("op<>")
assert isinstance(atom_op, grammar.Atom) and atom_op.word == "op"
assert isinstance(app_op, grammar.App) and len(app_op.children) == 0

# 2. Syntax error localization: Unterminated atom
try:
    grammar.parse("<unterminated")
    assert False, "Expected GrammarError"
except grammar.GrammarError as e:
    assert e.position == 13
    assert "unterminated" in e.reason.lower()

# 3. Unclosed application detection
try:
    grammar.parse("op<<0>")
    assert False, "Expected GrammarError"
except grammar.GrammarError as e:
    assert "unclosed application" in e.reason.lower()

# 4. Bare content rejection (no delimiters)
try:
    grammar.parse("bare_word")
    assert False, "Expected GrammarError"
except grammar.GrammarError as e:
    assert e.position == 0

print("Parser lookahead, prefix disambiguation, and error localization verified successfully!")

## 3. Diagnostic Probe (2026 Micro-Attention Check)

**Question:** What does Grammar 000 produce when parsing `<op>` versus `op<>`?

- **A:** Both produce the same application.
- **B:** `<op>` produces `Atom("op")`, while `op<>` produces `App(())`.
- **C:** `<op>` is an invalid reserved keyword.

*(Correct answer: **B**. Because `<op>` starts with `<`, the prefix `op` is inside the atom delimiters. `op<>` starts with the reserved prefix `op<`, creating an App.)*

In [ ]:
# Interactive Diagnostic Probe Check
student_choice = "B"  # Change to experiment
assert student_choice == "B", "Incorrect! Expected B: <op> is Atom, op<> is App"

# Mechanically verify the property
p1 = grammar.parse("<op>")
p2 = grammar.parse("op<>")
assert type(p1) is grammar.Atom and p1.word == "op"
assert type(p2) is grammar.App and p2.children == ()
assert type(p1) is not type(p2)

print("Diagnostic Probe: PASSED (<op> is Atom, op<> is App)")

## 4. Student Exercises (Ages 5 to 500)

1. **Exercise 1 (Age 5):** Can you tell the difference between a box labelled `<cat>` and an open box `op<<cat>>`?
2. **Exercise 2 (Age 15):** If you pass `"<unterminated"` to `grammar.parse()`, at what character position does the parser report the error? Test it below!
3. **Exercise 3 (Age 500):** Verify that the parser rejects unclosed applications like `op<<0>` and detects trailing characters after outer packets like `<0>extra`.

In [ ]:
# Exercise Solution & Verification
# 1. Parsing Law 1: Rejection of multiple outer packets
try:
    grammar.parse("<0><1>")
    assert False, "Expected GrammarError on multiple outer packets"
except grammar.GrammarError as e:
    assert "trailing content" in e.reason.lower() or "extra" in e.reason.lower() or e.position == 3

# 2. Trailing garbage after packet
try:
    grammar.parse("<0>extra")
    assert False, "Expected GrammarError on trailing garbage"
except grammar.GrammarError as e:
    assert e.position == 3

print("Student exercises verified successfully: trailing content rejection.")

## 5. Anchors & References

| Anchor Type | Target Link |
| :--- | :--- |
| **Curriculum Anchor** | [Grammar 000 Parser](../../../../omega_engine_symbol_game/domain/tower/grammar.py) |
| **Authority Effect** | `None` |
| **Parsing Law 6** | Maximal recognition of prefix `op<` vs `<` |